In [ ]:
# ======================================
# SmartChat Insight
#  Módulo de Predicción y Recomendación
# ======================================

# Importar librerías
import pandas as pd

In [ ]:

# -------------------------------
# 1. Cargar los archivos finales
# -------------------------------
clientes = pd.read_csv("../data/outputs/final/clientes_powerbi.csv", sep=";")
clientes_final = pd.read_csv("../data/outputs/final/clientes_final_powerbi.csv", sep=";")
productos = pd.read_csv("../data/outputs/final/productos_powerbi.csv", sep=";")
resumen = pd.read_csv("../data/outputs/final/resumen_clientes.csv", sep=";")

print("Archivos cargados correctamente")
clientes_final.head()

In [ ]:

# Convertir columnas de fecha
for col in ["primer_contacto", "ultimo_contacto"]:
    if col in clientes_final.columns:
        clientes_final[col] = pd.to_datetime(clientes_final[col], errors="coerce")

# Crear fecha de corte
fecha_corte = clientes_final["ultimo_contacto"].max().date()
print("Fecha de corte del reporte:", fecha_corte)

In [ ]:
# Reglas base
REGLAS = {
    "Frecuente": {
        "accion": "Mantener flujo de comunicación",
        "probabilidad_conversion": 0.75,
        "dias_para_contactar": 7
    },
    "Inactivo reciente": {
        "accion": "Mensaje de seguimiento",
        "probabilidad_conversion": 0.45,
        "dias_para_contactar": 2
    },
    "Perdido": {
        "accion": "Campaña de reactivación",
        "probabilidad_conversion": 0.18,
        "dias_para_contactar": 1
    },
}


In [ ]:
# Cliente nuevo = primer contacto en últimos 3 días
clientes_final["nuevo"] = clientes_final["primer_contacto"] >= (
    pd.to_datetime(fecha_corte) - pd.Timedelta(days=3)
)

# Asignar etiqueta "Nuevo"
clientes_final.loc[clientes_final["nuevo"], "estado"] = "Nuevo"

# Agregar reglas para "Nuevo"
REGLAS["Nuevo"] = {
    "accion": "Mensaje de bienvenida",
    "probabilidad_conversion": 0.90,
    "dias_para_contactar": 0
}

In [ ]:
def recomendar(row):
    estado = row["estado"]
    ultimo = row["ultimo_contacto"]

    regla = REGLAS.get(estado, None)
    if regla is None:
        return pd.Series({
            "accion_recomendada": "Revisar",
            "probabilidad_conversion": np.nan,
            "fecha_recomendada_contacto": None
        })
    
    accion = regla["accion"]
    prob = regla["probabilidad_conversion"]
    dias_para_contactar = regla["dias_para_contactar"]

    fecha_recomendada = ultimo + pd.Timedelta(days=dias_para_contactar)

    return pd.Series({
        "accion_recomendada": accion,
        "probabilidad_conversion": prob,
        "fecha_recomendada_contacto": fecha_recomendada.date()
    })

reco = clientes_final.apply(recomendar, axis=1)
clientes_final = pd.concat([clientes_final, reco], axis=1)

In [ ]:

def generar_mensaje(row):
    estado = row["estado"]
    nombre = row["user"] if "user" in row else "cliente"

    mensajes = {
        "Frecuente": f"Hola {nombre}, gracias por mantener el contacto 😊. ¿En qué podemos ayudarte hoy?",
        "Inactivo reciente": f"Hola {nombre}, hace unos días no conversamos. ¿Te sigo ayudando con tu pedido?",
        "Perdido": f"Hola {nombre}, tenemos novedades y promociones disponibles para ti. ¿Deseas verlas?",
        "Nuevo": f"¡Bienvenido {nombre}! Gracias por escribirnos 😊. ¿En qué podemos ayudarte hoy?"
    }

    return mensajes.get(estado, "Hola, ¿cómo podemos ayudarte?")

clientes_final["mensaje_sugerido"] = clientes_final.apply(generar_mensaje, axis=1)

In [ ]:

clientes_final[[
    "user", "estado", "dias_desde_ultimo",
    "accion_recomendada", "probabilidad_conversion",
    "fecha_recomendada_contacto", "mensaje_sugerido"
]].head(10)

In [ ]:

output_path = "../data/outputs/final/acciones_recomendadas.csv"
clientes_final.to_csv(output_path, index=False, sep=";")

print("Archivo exportado exitosamente:", output_path)